In [ ]:
import boto3
import pandas as pd
from io import StringIO
import os
from dotenv import load_dotenv
import numpy as np
import mlflow
import mlflow.sklearn
from datetime import datetime

load_dotenv()

# MLflow Configuration
mlflow.set_tracking_uri("http://localhost:5000")  # MLflow tracking server
mlflow.set_experiment("AQI_Weather_Prediction")


2025/10/22 18:28:27 INFO mlflow.tracking.fluent: Experiment with name 'AQI_Weather_Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1761139708023, experiment_id='1', last_update_time=1761139708023, lifecycle_stage='active', name='AQI_Weather_Prediction', tags={}>

In [4]:
# Your AWS credentials and bucket info
bucket_name = 'my-feature-store-data'
s3_key = 'pipeline-data/data.csv'  # Example: "pipeline-data/data.csv"

# Create an S3 client
s3 = boto3.client(
    's3',
    aws_access_key_id= os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

# Fetch the object from S3
response = s3.get_object(Bucket=bucket_name, Key=s3_key)

# Read the CSV content
csv_data = response['Body'].read().decode('utf-8')

# Convert to DataFrame
df = pd.read_csv(StringIO(csv_data))

# Done!
print(df.isnull().sum())

index                         0
aqi_index                     0
co                            0
no                            0
no2                           0
o3                            0
so2                           0
pm2_5                         0
pm10                          0
nh3                           0
temperature_2m                0
relative_humidity_2m          0
precipitation                 0
wind_speed_10m                0
wind_direction_10m            0
surface_pressure              0
dew_point_2m                  0
apparent_temperature          0
shortwave_radiation           0
et0_fao_evapotranspiration    0
year                          0
month                         0
day                           0
hour                          0
Calculated_AQI                0
dtype: int64


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split


# Step 4: Define target and features
# Step 4: Define targets and features
target_columns = ["aqi_index", "Calculated_AQI"]
targets = df[target_columns]
X = df.drop(columns=target_columns)

# Step 5: Final check for datetime columns
X = X.select_dtypes(exclude=["datetime64[ns]"])

# Step 6: Split
X_train, X_test, y_train, y_test = train_test_split(X, targets, test_size=0.2, random_state=42)


In [6]:
# Import necessary libraries
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

In [7]:
# Import additional required libraries
from io import BytesIO
from sklearn.multioutput import MultiOutputRegressor

# Step 3: Define Models with MultiOutput capability
models = {
    "Random_Forest": RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42),
    "Gradient_Boosting": MultiOutputRegressor(GradientBoostingRegressor(n_estimators=300, max_depth=3, random_state=42)),
    "Linear_Regression": LinearRegression(),
    "Ridge_Regression": Ridge(alpha=1.0),
    "SVR": MultiOutputRegressor(SVR()),
    "Neural_Network": MultiOutputRegressor(MLPRegressor(max_iter=200, random_state=42))
}

# -----------------------
# Step 4: Train and Evaluate with MLflow Tracking
results = []
best_model = None
best_model_name = None
best_avg_rmse = float("inf")

for model_name, model in models.items():
    # Start MLflow run for each model
    with mlflow.start_run(run_name=f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        print(f"Training {model_name}...")
        
        # Log model parameters
        if hasattr(model, 'get_params'):
            mlflow.log_params(model.get_params())
        
        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics for each target
        model_results = {"Model": model_name}
        rmse_scores = []
        
        for i, col in enumerate(target_columns):
            rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))
            mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
            r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
            
            rmse_scores.append(rmse)
            model_results[f"RMSE_{col}"] = rmse
            model_results[f"MAE_{col}"] = mae
            model_results[f"R²_{col}"] = r2
            
            # Log metrics to MLflow for each target
            mlflow.log_metric(f"rmse_{col}", rmse)
            mlflow.log_metric(f"mae_{col}", mae)
            mlflow.log_metric(f"r2_{col}", r2)
            
            print(f"  Target: {col}")
            print(f"    RMSE: {rmse}")
            print(f"    MAE: {mae}")
            print(f"    R²: {r2}")
        
        # Calculate average RMSE across all targets
        avg_rmse = np.mean(rmse_scores)
        avg_mae = np.mean([model_results[f"MAE_{col}"] for col in target_columns])
        avg_r2 = np.mean([model_results[f"R²_{col}"] for col in target_columns])
        
        model_results["Avg_RMSE"] = avg_rmse
        results.append(model_results)
        
        # Log aggregate metrics
        mlflow.log_metric("avg_rmse", avg_rmse)
        mlflow.log_metric("avg_mae", avg_mae)
        mlflow.log_metric("avg_r2", avg_r2)
        
        # Log model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        # Add tags
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("targets", str(target_columns))
        
        print(f"  Average RMSE: {avg_rmse}\n")
        
        if avg_rmse < best_avg_rmse:
            best_avg_rmse = avg_rmse
            best_model = model
            best_model_name = model_name

# -----------------------
# Step 5: Register Best Model in MLflow Model Registry
print(f"\nBest model: {best_model_name} (Average RMSE = {best_avg_rmse:.2f})")

# Log and register the best model
with mlflow.start_run(run_name=f"BEST_{best_model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    mlflow.sklearn.log_model(
        best_model, 
        "best_model",
        registered_model_name="AQI_Weather_Best_Model"
    )
    mlflow.log_metric("best_avg_rmse", best_avg_rmse)
    mlflow.set_tag("best_model", best_model_name)
    mlflow.set_tag("production_ready", "true")

# -----------------------
# Step 6: Summary
results_df = pd.DataFrame(results)
print("\nSummary of Model Performance:")
print(results_df)


Training Random_Forest...
  Target: aqi_index
    RMSE: 0.34738908398334095
    MAE: 0.21662652854163308
    R²: 0.8732577172488808
  Target: aqi_index
    RMSE: 0.34738908398334095
    MAE: 0.21662652854163308
    R²: 0.8732577172488808
  Target: Calculated_AQI
    RMSE: 8.283295183093642
    MAE: 1.5675551193467248
    R²: 0.9934348230739033
  Target: Calculated_AQI
    RMSE: 8.283295183093642
    MAE: 1.5675551193467248
    R²: 0.9934348230739033


2025/10/22 18:28:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/22 18:29:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/22 18:29:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Average RMSE: 4.315342133538492

🏃 View run Random_Forest_20251022_182837 at: http://localhost:5000/#/experiments/1/runs/d64454918f524ed8afd02090b24acdde
🧪 View experiment at: http://localhost:5000/#/experiments/1
Training Gradient_Boosting...
Training Gradient_Boosting...
  Target: aqi_index
    RMSE: 0.049357553156314043
    MAE: 0.017063657148444347
    R²: 0.9974414351228401
  Target: aqi_index
    RMSE: 0.049357553156314043
    MAE: 0.017063657148444347
    R²: 0.9974414351228401
  Target: Calculated_AQI
    RMSE: 6.217464344023747
    MAE: 3.082012149436491
    R²: 0.9963011496147057
  Target: Calculated_AQI
    RMSE: 6.217464344023747
    MAE: 3.082012149436491
    R²: 0.9963011496147057


2025/10/22 18:29:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/22 18:29:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/22 18:29:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Average RMSE: 3.1334109485900306

🏃 View run Gradient_Boosting_20251022_182920 at: http://localhost:5000/#/experiments/1/runs/968713dc98af432984352ed273930568
🧪 View experiment at: http://localhost:5000/#/experiments/1
Training Linear_Regression...
Training Linear_Regression...
  Target: aqi_index
    RMSE: 0.583291746494057
    MAE: 0.47149577566508766
    R²: 0.6426770979739415
  Target: aqi_index
    RMSE: 0.583291746494057
    MAE: 0.47149577566508766
    R²: 0.6426770979739415
  Target: Calculated_AQI
    RMSE: 82.40410746625577
    MAE: 58.440604348179235
    R²: 0.35026258261362575
  Target: Calculated_AQI
    RMSE: 82.40410746625577
    MAE: 58.440604348179235
    R²: 0.35026258261362575


...

  Target: aqi_index
    RMSE: 3.8729945789663858
    MAE: 3.0409402173481013
    R²: -14.753714411629645
  Target: Calculated_AQI
    RMSE: 38.24047080130447
    MAE: 22.097352711422964
    R²: 0.8600778018544479
  Target: Calculated_AQI
    RMSE: 38.24047080130447
    MAE: 22.097352711422964
    R²: 0.8600778018544479


2025/10/22 18:30:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/22 18:30:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/22 18:30:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Average RMSE: 21.056732690135426

🏃 View run Neural_Network_20251022_183022 at: http://localhost:5000/#/experiments/1/runs/36b92fa6a5874faa806d06b4dedb18c3
🧪 View experiment at: http://localhost:5000/#/experiments/1

Best model: Gradient_Boosting (Average RMSE = 3.13)


2025/10/22 18:30:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/22 18:30:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/22 18:30:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'AQI_Weather_Best_Model'.
Successfully registered model 'AQI_Weather_Best_Model'.
2025/10/22 18:30:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AQI_Weather_Best_Model, version 1
2025/10/22 18:30:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AQI_Weather_Best_Model, version 1
Created version '1' of model 'A

🏃 View run BEST_Gradient_Boosting_20251022_183034 at: http://localhost:5000/#/experiments/1/runs/78159ca540b24e4c94e78bd9b63dcbe6
🧪 View experiment at: http://localhost:5000/#/experiments/1

Summary of Model Performance:
               Model  RMSE_aqi_index  MAE_aqi_index  R²_aqi_index  \
0      Random_Forest        0.347389       0.216627      0.873258   
1  Gradient_Boosting        0.049358       0.017064      0.997441   
2  Linear_Regression        0.583292       0.471496      0.642677   
3   Ridge_Regression        0.583469       0.470926      0.642460   
4                SVR        0.456223       0.324563      0.781403   
5     Neural_Network        3.872995       3.040940    -14.753714   

   RMSE_Calculated_AQI  MAE_Calculated_AQI  R²_Calculated_AQI   Avg_RMSE  
0             8.283295            1.567555           0.993435   4.315342  
1             6.217464            3.082012           0.996301   3.133411  
2            82.404107           58.440604           0.350263  41.4937

In [8]:
import joblib
import json
import sklearn
import numpy as np
from io import BytesIO

S3_MODEL_KEY = "models/best_model.pkl"
S3_METADATA_KEY = "models/best_model_metadata.json"

def upload_model_to_s3(model, bucket_name, s3_client):
    # --- Save model to BytesIO buffer ---
    model_buffer = BytesIO()
    joblib.dump(model, model_buffer)
    model_buffer.seek(0)
    s3_client.upload_fileobj(model_buffer, Bucket=bucket_name, Key=S3_MODEL_KEY)
    print(f"Model uploaded to s3://{bucket_name}/{S3_MODEL_KEY}")

    # --- Save version metadata ---
    metadata = {
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "model_type": type(model).__name__,
    }

    metadata_buffer = BytesIO()
    metadata_buffer.write(json.dumps(metadata).encode("utf-8"))
    metadata_buffer.seek(0)
    s3_client.upload_fileobj(metadata_buffer, Bucket=bucket_name, Key=S3_METADATA_KEY)
    print(f"Metadata uploaded to s3://{bucket_name}/{S3_METADATA_KEY}")


In [9]:
upload_model_to_s3(best_model, bucket_name, s3)

Model uploaded to s3://my-feature-store-data/models/best_model.pkl
Metadata uploaded to s3://my-feature-store-data/models/best_model_metadata.json
Metadata uploaded to s3://my-feature-store-data/models/best_model_metadata.json
